# Day 39: NLP Patterns & Aspect-Level Sentiment

### Q1: The 5 Hardest NLP Patterns in Indian E-Commerce Reviews

In [ ]:
import pandas as pd
import re
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def analyze_baseline(text):
    return TextBlob(text).sentiment.polarity

def analyze_vader(text):
    return analyzer.polarity_scores(text)['compound']

reviews = {
    'negation': 'not bad at all',
    'sarcasm': 'Wow great! Broke on day 1',
    'code_mixing': 'Product bahut accha hai lekin delivery late thi',
    'implicit': 'Returned it within 2 hours',
    'comparative': 'Way better than my previous Samsung'
}

for pattern, text in reviews.items():
    print(f"[{pattern.upper()}] Text: '{text}'")
    print(f"  -> Baseline (TextBlob) polarity: {analyze_baseline(text):.2f}")
    print(f"  -> Robust (VADER) compound score: {analyze_vader(text):.2f}")
    print()
    
# Explanation of failures and solutions:
# (a) Negation: Baseline sees 'bad' as negative (-0.7). Solution: VADER handles simple 'not' modifiers to flip sentiment.
# (b) Sarcasm: Baseline sees 'great' (+0.8). Solution: Context-based sequence models (e.g., BERT) or specific syntactical rule checkers. VADER also struggles here.
# (c) Code-mixing: Baseline ignores non-English words. Solution: mBERT or transliteration APIs before analysis.
# (d) Implicit: No explicit sentiment words. Baseline=0.0. Solution: Domain-specific classifiers mapping actions ("returned") to negative sentiment.
# (e) Comparative: Confusing subject reference. Solution: Dependency parsing or Aspect-Based Sentiment Analysis (ABSA).


### Q2: Review-Level vs Aspect-Level Classifier

In [ ]:
import spacy
from collections import defaultdict

# Setup NLP. Using small English core web model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

# (a) Why is aspect-level harder?
# Answer: A single text can have conflicting sentiments towards different features. 
# Aggregation causes information loss. Dependencies between specific adverbs/adjectives 
# and their target nouns must be accurately parsed.

# (b) How to improve from 71% to 80%+?
# Answer: 
# 1. Use pre-trained transformers specifically for ABSA (e.g., PyABSA / DeBERTa).
# 2. Use BIO (Begin, Inside, Outside) tagging for aspect term extraction.
# 3. Augment data using syntactic dependency structures.

# (c) Extract aspect-sentiment pairs
review = 'Amazing camera quality but the battery is atrocious and customer support was unhelpful.'
doc = nlp(review)

aspect_pairs = []
# Very simple dependency based extraction (Heuristic)
for chunk in doc.noun_chunks:
    aspect = chunk.text
    # Finding modifiers for this noun
    sentiment_words = []
    for token in chunk.root.head.children:
        if token.pos_ == 'ADJ':
            sentiment_words.append(token.text)
    
    # Check parent adjectives (e.g. Amazing camera)
    for token in chunk:
        if token.pos_ == 'ADJ' and token.dep_ == 'amod':
            sentiment_words.append(token.text)
            
    # Check adjectival complement (e.g. battery is atrocious)
    if chunk.root.dep_ == 'nsubj' and chunk.root.head.pos_ in ['AUX', 'VERB']:
        for child in chunk.root.head.children:
            if child.dep_ in ['acomp', 'attr', 'amod'] or child.pos_ == 'ADJ':
                sentiment_words.append(child.text)

    if sentiment_words:
        aspect_pairs.append({
            'aspect': aspect,
            'sentiment_indicators': ", ".join(set(sentiment_words))
        })

print("Extracted Aspect Pairs:")
for pair in aspect_pairs:
    print(f" - Aspect: '{pair['aspect']}' | Sentiment Text: '{pair['sentiment_indicators']}'")

# Based on this:
# 'camera quality' -> Amazing (Positive)
# 'the battery' -> atrocious (Negative)
# 'customer support' -> unhelpful (Negative)
